# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane falls into the ranking/scoring category because it will rank pages in order of priority based on refresh value and decline strength. It would also use a classifier or regressor to predict decline severity and recovery probability.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The pieces of the final score are:
- Decline severity - predict the loss that the page will experience: clicks_prev_30d - clicks_last_30d
- Recovery probability - predict the probability of recovery, or the chance of improvement from current window to next.
Both are observed outcomes and reflect the current page's outcome.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@K for each piece of the final score would tell us how many pages actually declined/recovered. When regression is used, we can also introduce a mean absolute error metric between the actual and predicted values. "Good" would simply mean beating the baseline score (the hand rule and potentially random forest model)

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

import pandas as pd
csv_df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [12]:
df = (
    csv_df[(csv_df["impressions_90d"] > 0) & (csv_df["content_age_days"] >= 90)]
    .drop_duplicates("content_id")
)

df["severity_target"] = df["clicks_prev_30d"] - df["clicks_last_30d"]
df["recovery_target"] = pd.NA  # requires future data

print(f"Lane slice shape: {df.shape}")
print()
print("One row = one page with enough traffic/history. Listed potential features and responses")
print(df[[
    "content_id", "client_id", "content_type",
    "impressions_90d", "clicks_90d", "avg_position", "content_age_days",
    "severity_target", "recovery_target"
]].head(3).to_string(index=False))

Lane slice shape: (30000, 46)

One row = one page with enough traffic/history. Listed potential features and responses
          content_id         client_id    content_type  impressions_90d  clicks_90d  avg_position  content_age_days  severity_target recovery_target
content_304f48230142 client_f369cb89fc keyword article             3803          29          10.6               187               11            <NA>
content_a1fb4e703a9e client_4e07408562 keyword article            15320           7          20.3               445               -1            <NA>
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          11          36.5               141                2            <NA>


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single feature rule cannot cleanly separate "worth reviewing" and "not". For example, if we flag all pages where the trend direction is down, we are not showing the severity of the decline and not considering the recovery potential.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.